# Review gp_collab_hazel results

Same structure as DOPE-MURI's `02_review_results.ipynb`, reading runs produced
by this package. Set `RUNS` below to the folder holding the finished tasks. The
first cell exports them into the layout `load_results` reads, then loads them;
nothing here launches training.

Needs `numpy pandas scipy scikit-learn matplotlib` (and `ipywidgets` for the
control panel). **No PyTorch** -- review is CPU-only and model-free.

Put this notebook and `results.py` in the `gp_collab_hazel` folder, next to
`gpc/` and `inputs/`.

In [ ]:
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
from gpc.results import (prepare, load_results, lolo_report, plot_parity,
                     plot_lolo_comparison, grouped_predictions)

ROOT = Path.cwd().resolve()
RUNS = ROOT / "runs/experiment_v1_oldPCS"
# The submission scripts write to runs/<run_tag>/; if RUNS holds exactly one
# such folder it is found automatically. This runs/ holds three, all given
# the _oldPCS suffix on 2026-09-18: experiment_v1_oldPCS (n_epochs=400),
# experiment_v1_pilot_oldPCS (n_epochs=10, a smoke test) and ard_v2_oldPCS
# (ard=true) -- so one is named explicitly above. In all three the
# pc_scores/pc_scores_long rows use the superseded loading_weighted
# definition; see PC_SCORES_CAVEAT.md in each folder. Every other feature
# section is unaffected. The replacement run is tagged runs/pcs_v3.

# Builds RUNS/hazel_export/collected the first time; pass refresh=True after
# adding or rerunning tasks. Methods are relabelled to hazel_gp's names
# (iid_stratified_8 -> iid_matched, kfold_stratified_5 -> kfold) so the
# arguments below match the original notebook; the folds stay ligand-stratified.
EXPORT = prepare(RUNS, bundle=ROOT / "inputs", refresh=False)

tables, collection = load_results(EXPORT)
print("Complete benchmark:", collection["complete"])
print("Completed tasks:", collection["completed_tasks"], "/", collection["expected_tasks"])
display(tables["summary"])

### Inspect predictions

Pick a model, method and held-out ligand; the parity plot and that split's
metrics redraw. Needs `ipywidgets`; skip this cell if it is not installed.

In [ ]:
from gpc.results import review_controls

# Parity-plot heading. None keeps the default "Model | method[| ligand]";
# otherwise a str.format template with `model`, `method`, `reference_group`
# fields, e.g. "{model} on {method} ({reference_group})".
TITLE = None
display(review_controls(tables, title=TITLE))

### LOLO reported three ways

`per_ligand` scores each held-out ligand on its own rows, `mean_across_ligands`
averages those, and `pooled_over_folds` concatenates all eight folds and scores
once. Pooled and averaged values are **not** interchangeable: a per-ligand R² is
measured against that ligand's own (smaller) variance, so the averaged figure is
systematically harsher.

In [ ]:
# Start from an empty figure registry. The inline backend re-renders every
# figure still open at the end of a cell, so anything leaked by an earlier
# run of this kernel would be drawn again here on top of this cell's own.
plt.close("all")

METRIC = "r2"
report = lolo_report(tables["predictions"], tables["metrics_by_split"])
display(report[["model", "view", "scope", "n_test", "r2", "rmse", "mae", "kendall_tau"]].round(4))

fig_ligands = plot_lolo_comparison(tables["metrics_by_split"], metric=METRIC)
display(fig_ligands)
plt.close(fig_ligands)

### Compare the three evaluation methods

LOLO against the two in-distribution controls. The stratified 8-fold has LOLO's
exact fold geometry (2688/384) with every ligand present in training, so the gap
between them is the cost of meeting an unseen ligand.

In [ ]:
from gpc.results import model_label, method_label, _model_order
# Start from an empty figure registry. The inline backend re-renders every
# figure still open at the end of a cell, so anything leaked by an earlier
# run of this kernel would be drawn again here on top of this cell's own.
plt.close("all")

summary = tables["summary"]
pooled = summary[summary.aggregation == "pooled_predictions"]
# One canonical model order (gpc.results.MODEL_ORDER) for the table and the
# chart, and the same one in the hazel and rxnpredict reviews, so the two
# read left-to-right identically. pivot() alone would sort alphabetically.
order = _model_order(pooled)
display(pooled.pivot(index="model", columns="method", values=["r2", "rmse"])
              .reindex(order).round(4))

# Grouped bars, not a line. The x axis is five unrelated feature sets, so a line
# joining them would imply an ordering and a continuum that do not exist; bars
# also match plot_lolo_comparison above. Heights are the POOLED figures, so they
# do not equal the mean of that model's per-fold bars in the LOLO chart.
table = pooled.pivot(index="model", columns="method", values=METRIC)
table = table.reindex(index=order)                # canonical order, not alphabetical
table = table[[m for m in ("lolo", "iid_matched", "kfold") if m in table.columns]]
table.index = [model_label(m) for m in table.index]
table.columns = [method_label(m) for m in table.columns]

unit = " (% yield)" if METRIC in ("rmse", "mae") else ""
ax = table.plot.bar(figsize=(9, 4.5), width=0.78, edgecolor="white", linewidth=0.6,
                    color=["#c0392b", "#5b8db8", "#a8c8e0"])
ax.set(xlabel="", ylabel=f"pooled {METRIC.upper()}{unit}")
ax.set_ylim(0, table.to_numpy().max() * 1.12)
ax.tick_params(axis="x", rotation=15)
ax.legend(title="Evaluation method", fontsize=8, title_fontsize=8, frameon=False,
          loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_title(f"Pooled out-of-fold {METRIC.upper()} by feature section", fontsize=11)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=7, padding=2)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.set_axisbelow(True)

fig_methods = ax.figure
fig_methods.tight_layout()
display(fig_methods)
plt.close(fig_methods)

In [ ]:
grouped_iid = grouped_predictions(tables["predictions"], model="selected_2", method="iid_matched")
print("Reference groups:", list(grouped_iid))
# The stratified 8-fold is a single non-repeating partition with every ligand in
# every fold, so it has no per-ligand reference group and returns one group keyed
# "". Use method="lolo" to get the eight held-out ligands instead:
grouped_lolo = grouped_predictions(tables["predictions"], model="selected_2", method="lolo")
print("LOLO groups:", list(grouped_lolo))
# grouped_lolo["SPhos"]["y_pred"] is a list of arrays, one per split.

### Feature importance (ARD)

Needs a run trained with `"ard": true` under `gp` in the run config. An isotropic
run (`ard: false`) learns ONE lengthscale for the whole feature block, so there is
nothing per-feature inside it; the cell below says so and names the fix rather than
inventing a number.

`relevance` is 1 / lengthscale. An RBF varies fastest along its shortest
lengthscales, so a large relevance means the fit leans on that column -- and a
feature the GP pushed out to a huge lengthscale is flat along that axis, i.e. it has
effectively been switched off. The ranking holds *within one model*: numeric
descriptors are standardized on the training fold while one-hots stay raw 0/1
indicators, and two models' kernels share no scale, so compare the ordering and not
the magnitudes.

Whiskers are the fold quartiles; a long whisker means the eight folds disagree and
that position is soft. Under LOLO a ligand's one-hot column only exists in the folds
where that ligand was in **training**, so for `ligand_ohe` those columns are scored
on 7 of 8 folds and their labels say so -- that is the split doing its job, not a
bug.

In [ ]:
from gpc.results import rank_features, plot_feature_importance
# Start from an empty figure registry. The inline backend re-renders every
# figure still open at the end of a cell, so anything leaked by an earlier
# run of this kernel would be drawn again here on top of this cell's own.
plt.close("all")

IMPORTANCE_MODEL = "selected_5"
LIGAND_ONLY      = True   # True -> ligand descriptors only; False -> every encoded feature
TOP_N            = 20     # rows in the table and bars in the chart; None for all of them
BUNDLE           = ROOT / "inputs"   # the prepared inputs cell 1 exported from

# RUNS, not EXPORT: lengthscales live in each task's own scores.json and the
# collected review tables do not carry them. Indices are resolved to feature NAMES
# per fold, because each LOLO fold drops a different ligand one-hot, so position 7
# is a different column in a different fold.
fig_importance = None
try:
    ranked = rank_features(RUNS, BUNDLE, IMPORTANCE_MODEL, method="lolo",
                           ligand_only=LIGAND_ONLY, top=TOP_N)
    display(ranked.round(4))
    fig_importance = plot_feature_importance(RUNS, BUNDLE, IMPORTANCE_MODEL, method="lolo",
                                             ligand_only=LIGAND_ONLY, top=TOP_N)
    display(fig_importance)
    plt.close(fig_importance)
except ValueError as exc:
    # An isotropic run has no per-feature lengthscales to show. Printed, not raised,
    # so the message is read and the rest of the notebook still runs.
    print(f"No per-feature relevance for {IMPORTANCE_MODEL}: {exc}")

In [ ]:
import pandas as pd
from gpc.results import rank_features, _model_order

# Every model in this run at once, ligand features only, so comparing them does not
# mean editing the toggles above five times. Relevances are NOT comparable between
# models -- each fit has its own kernel and its own scale -- so read each model's
# block as a ranking and ignore how one model's numbers sit against another's.
SUMMARY_TOP = 5

frames, unavailable = [], []
for model in _model_order(tables["summary"]):
    try:
        part = rank_features(RUNS, BUNDLE, model, method="lolo",
                             ligand_only=True, top=SUMMARY_TOP)
    except ValueError as exc:
        unavailable.append((model, str(exc)))
        continue
    frames.append(part.assign(model=model)[["model", "feature", "median_relevance",
                                            "median_lengthscale", "folds", "n_folds"]])

if frames:
    display(pd.concat(frames, ignore_index=True).round(4))
for position, (model, message) in enumerate(unavailable):
    # The full message once -- it names the fix -- then the leading clause of each
    # other one, which is enough to see whether it failed for the same reason.
    print(f"{model}: {message if position == 0 else message.split(' -- ')[0]}")

### Performance vs train/test distance in feature space

Each LOLO fold holds out one ligand, so every point here is one ligand: its fold's
score against how far it sat from the ligands the GP actually saw. Distance is
Euclidean in standardized ligand-descriptor space -- the `num__` columns only,
scaled by that fold's OWN training scaler, so the held-out ligand never touches the
scaler and the geometry is the one that fold's GP was fitted in. Condition one-hots
are left out: every ligand is run over the same grid, so they would measure the
design and not the chemistry.

Two distances, because they can disagree. `centroid_distance` is to the mean of the
training ligands; `nearest_distance` is to the closest single one, and is usually
the better read on extrapolation -- a ligand can sit near the training average while
resembling nothing in the set.

Eight folds means eight points (four in the rxnpredict review), so the Spearman rho
is suggestive, not conclusive: one ligand moves it visibly and the p-values are
decoration at this n. Read its sign against the metric -- for `rmse` positive rho is
the expected "farther is worse", for `r2` it is negative rho. `ligand_ohe` is
refused here rather than plotted: a one-hot ligand block places every ligand the
same distance from every other, so it has no descriptor geometry to measure.

In [ ]:
from gpc.results import distance_vs_performance, plot_distance_vs_performance
# Start from an empty figure registry. The inline backend re-renders every
# figure still open at the end of a cell, so anything leaked by an earlier
# run of this kernel would be drawn again here on top of this cell's own.
plt.close("all")

DISTANCE_MODEL  = "selected_5"   # needs numeric descriptors; ligand_ohe has none
DISTANCE_METRIC = "rmse"         # any metrics_by_split column: rmse, mae, r2, kendall_tau
BUNDLE          = ROOT / "inputs"

fig_distance = None
try:
    distances = distance_vs_performance(RUNS, BUNDLE, DISTANCE_MODEL, method="lolo",
                                        metric=DISTANCE_METRIC)
    display(distances.round(4))
    # The two correlations live in .attrs rather than in a column: they describe the
    # whole table, and a constant repeated down eight rows reads as a per-fold number.
    for name, (rho, p) in distances.attrs["spearman"].items():
        print(f"{name:>8} distance vs {DISTANCE_METRIC}: "
              + ("Spearman rho n/a" if rho is None else
                 f"Spearman rho = {rho:+.2f}  (p = {p:.3f}, n = {len(distances)} ligands)"))
    fig_distance = plot_distance_vs_performance(RUNS, BUNDLE, DISTANCE_MODEL, method="lolo",
                                                metric=DISTANCE_METRIC)
    display(fig_distance)
    plt.close(fig_distance)
except ValueError as exc:
    print(f"No distance/performance comparison for {DISTANCE_MODEL}: {exc}")

### Final export

Nothing is written until `SAVE = True`.

In [ ]:
SAVE = False
EXPORT_DIR = ROOT / "exports/local_review_v1"
FIGURE_FORMATS = ["png", "pdf"]
TABLES_TO_SAVE = ["summary", "metrics_by_split", "predictions"]

if SAVE:
    EXPORT_DIR.mkdir(parents=True, exist_ok=False)
    for name in TABLES_TO_SAVE:
        tables[name].to_csv(EXPORT_DIR / f"{name}.csv", index=False)
    report.to_csv(EXPORT_DIR / "lolo_report.csv", index=False)
    # globals().get: a section that was never run leaves its name undefined and a
    # section this run cannot support leaves its figure None -- neither may break
    # the export, so both are dropped here.
    figures = {"lolo_by_ligand": fig_ligands, "method_comparison": fig_methods,
               "feature_importance": globals().get("fig_importance"),
               "distance_vs_performance": globals().get("fig_distance")}
    figures = {name: fig for name, fig in figures.items() if fig is not None}
    for name, fig in figures.items():
        for fmt in FIGURE_FORMATS:
            fig.savefig(EXPORT_DIR / f"{name}.{fmt}", dpi=300, bbox_inches="tight")
    print("Saved:", EXPORT_DIR)
else:
    print("Preview only. Set SAVE=True when ready.")